# 05 · final cost comparison — the only notebook that reads the test set

Every approach froze its hyper-parameters in `lab/results/`, chosen on **validation**
by cost. Each is now refit on **train + validation** and scored once on **test**.

If you edit an approach after reading this, the test set is spent and the next honest
number needs new data.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
import dataset as D

frame = D.build("claude")
train, validation, test = D.split(frame)
COLS = D.features(frame)
print("\ncost model:  stolen $%.2f/pct   wasted $%.2f/pct   ratio %.0f:1"
      % (D.COST_STOLEN, D.COST_WASTED, D.COST_STOLEN / D.COST_WASTED))
print("the cost-optimal quantile is therefore %.4f" % D.optimal_quantile())
frozen = D.load_all()
fit_frame = pd.concat([train, validation]).sort_values("ts")
print("\napproaches: %s" % ", ".join(f["name"] for f in frozen))
print("refit on %d rows (train+validation), scored on %d test rows"
      % (len(fit_frame), len(test)))

12075 anchors over 41.9 days, embargo 300 min at each seam
  train        7185 rows  08-24 17:55 -> 09-18 16:35
  validation   2355 rows  09-18 21:40 -> 09-27 01:50
  test         2415 rows  09-27 06:55 -> 10-05 16:05
  dropped to the embargo: 120

cost model:  stolen $3.20/pct   wasted $0.32/pct   ratio 10:1
the cost-optimal quantile is therefore 0.9091

approaches: 000_do_nothing, 00_constant, 01_recent_rate, 02_state_clock, 03_sklearn_quantile, 04_hierarchical_gaussian
refit on 9540 rows (train+validation), scored on 2415 test rows


In [2]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def build_predictor(outer, N):
    kind = outer["kind"]
    cfg = dict(outer["per_horizon"][str(N)])
    cfg["kind"] = kind
    if kind == "constant":
        pct = cfg["pct"]
        return lambda f: np.full(len(f), pct)
    if kind == "recent_rate":
        L, mult = cfg["lookback"], cfg["multiplier"]
        return lambda f: f["burn_%dm" % L].values / L * N * mult
    if kind == "state_clock":
        w, min_n, q = cfg["band_hours"], cfg["min_n"], cfg["q"]
        y = fit_frame["y%d" % N].values
        s = D.state(fit_frame); b = (fit_frame["hour"].values // w) * w
        ok = ~np.isnan(y)
        cell, per_state = {}, {}
        overall = float(np.percentile(y[ok], q * 100))
        for st in D.STATES:
            m = ok & (s == st)
            if m.sum() >= min_n:
                per_state[st] = float(np.percentile(y[m], q * 100))
            for bd in np.unique(b):
                mm = m & (b == bd)
                if mm.sum() >= min_n:
                    cell[(st, int(bd))] = float(np.percentile(y[mm], q * 100))
        def f_(f):
            s2 = D.state(f); b2 = (f["hour"].values // w) * w
            return np.array([cell.get((st, int(bd)), per_state.get(st, overall))
                             for st, bd in zip(s2, b2)])
        return f_
    if kind == "hist_quantile":
        kw = {k: v for k, v in cfg.items() if k not in ("kind", "q")}
        m = ~np.isnan(fit_frame["y%d" % N].values)
        g = HistGradientBoostingRegressor(loss="quantile", quantile=cfg["q"],
                                          random_state=0, **kw)
        g.fit(fit_frame.loc[m, COLS], fit_frame.loc[m, "y%d" % N])
        return lambda f: g.predict(f[COLS])
    if kind == "gaussian_ridge":
        m = ~np.isnan(fit_frame["y%d" % N].values)
        pipe = make_pipeline(StandardScaler(), Ridge(alpha=cfg["alpha"]))
        pipe.fit(fit_frame.loc[m, COLS], fit_frame.loc[m, "y%d" % N])
        resid = (fit_frame.loc[m, "y%d" % N] - pipe.predict(fit_frame.loc[m, COLS])).values
        if cfg["sigma_per_state"]:
            s = D.state(fit_frame.loc[m])
            sigma = {st: float(np.std(resid[s == st])) if (s == st).sum() > 20
                     else float(np.std(resid)) for st in D.STATES}
        else:
            sigma = {st: float(np.std(resid)) for st in D.STATES}
        z = cfg["z"]
        return lambda f: pipe.predict(f[COLS]) + z * np.array([sigma[st] for st in D.state(f)])
    raise ValueError(kind)

predictors = {f["name"]: {N: build_predictor(f["config"], N) for N in D.HORIZONS}
              for f in frozen}
configs = {f["name"]: f["config"] for f in frozen}
for n, c in configs.items():
    print("  %s" % n)
    for N in D.HORIZONS:
        print("      N=%-4d %s" % (N, c["per_horizon"][str(N)]))

  000_do_nothing
      N=30   {'pct': 100.0}
      N=60   {'pct': 100.0}
      N=120  {'pct': 100.0}
      N=180  {'pct': 100.0}
      N=300  {'pct': 100.0}
  00_constant
      N=30   {'pct': 0.0}
      N=60   {'pct': 0.0}
      N=120  {'pct': 0.0}
      N=180  {'pct': 13.0}
      N=300  {'pct': 22.0}
  01_recent_rate
      N=30   {'lookback': 120, 'multiplier': 2.0}
      N=60   {'lookback': 30, 'multiplier': 2.0}
      N=120  {'lookback': 60, 'multiplier': 3.0}
      N=180  {'lookback': 30, 'multiplier': 5.0}
      N=300  {'lookback': 30, 'multiplier': 5.0}
  02_state_clock
      N=30   {'band_hours': 2, 'min_n': 100, 'q': 0.75}
      N=60   {'band_hours': 6, 'min_n': 100, 'q': 0.85}
      N=120  {'band_hours': 6, 'min_n': 10, 'q': 0.85}
      N=180  {'band_hours': 6, 'min_n': 10, 'q': 0.85}
      N=300  {'band_hours': 4, 'min_n': 30, 'q': 0.75}
  03_sklearn_quantile
      N=30   {'q': 0.909, 'max_depth': 3, 'max_iter': 200, 'learning_rate': 0.05}
      N=60   {'q': 0.85, 'max_depth'

## Cost on test, per horizon

In [3]:
rows = []
for name, per_N in predictors.items():
    for N in D.HORIZONS:
        p = per_N[N](test)
        c = D.cost(test["y%d" % N].values, p)
        rows.append({"model": name, "N": N, "cost_usd": c["per_anchor"],
                     "stolen_usd": c["stolen_usd"], "wasted_usd": c["wasted_usd"],
                     "stolen_pct": c["stolen_pct"], "wasted_pct": c["wasted_pct"],
                     "mean_pred": float(np.nanmean(np.clip(p, 0, 100)))})
R = pd.DataFrame(rows)
for N in D.HORIZONS:
    sub = R[R["N"] == N].sort_values("cost_usd")
    print("\nN = %d min" % N)
    print(sub[["model", "cost_usd", "stolen_usd", "wasted_usd", "stolen_pct",
               "wasted_pct", "mean_pred"]].round(3).to_string(index=False))


N = 30 min
                   model  cost_usd  stolen_usd  wasted_usd  stolen_pct  wasted_pct  mean_pred
     03_sklearn_quantile     0.701       0.437       0.264       0.137       0.825      4.474
          01_recent_rate     0.973       0.758       0.215       0.237       0.673      2.141
04_hierarchical_gaussian     0.985       0.969       0.017       0.303       0.052      3.790
          02_state_clock     1.025       1.016       0.009       0.317       0.029      1.119
             00_constant     1.058       1.058       0.000       0.331       0.000      0.000
          000_do_nothing    28.694       0.000      28.694       0.000      89.669    100.000

N = 60 min
                   model  cost_usd  stolen_usd  wasted_usd  stolen_pct  wasted_pct  mean_pred
          02_state_clock     1.849       1.434       0.415       0.448       1.297      3.299
          01_recent_rate     2.589       1.728       0.861       0.540       2.690      4.308
04_hierarchical_gaussian     2.637  

## The headline: mean cost across horizons

`stolen_pct` and `wasted_pct` are percent of a 5-hour meter per anchor. They are **not**
multiplied up into a weekly bill: anchors overlap, and more importantly the *weekly*
meter caps total consumption at 8.85 units, so treating every 5-hour stretch of wall
clock as a spendable window overcounts by about 4x. Ranking is done on `cost_usd`; the
real weekly figure needs the budget stage's back-fill in the loop, which is a different
experiment.

In [4]:
agg = (R.groupby("model")[["cost_usd", "stolen_usd", "wasted_usd",
                            "stolen_pct", "wasted_pct"]]
       .mean().sort_values("cost_usd").round(3))
print(agg.to_string())
base = agg.loc["01_recent_rate", "cost_usd"]
print("\nimprovement over today's engine (01_recent_rate):")
for name, r in agg.iterrows():
    print("  %-26s %+6.1f%%" % (name, (r["cost_usd"] - base) / base * 100))

                          cost_usd  stolen_usd  wasted_usd  stolen_pct  wasted_pct
model                                                                             
02_state_clock               6.958       5.959       0.999       1.862       3.123
01_recent_rate               7.033       5.748       1.285       1.796       4.017
03_sklearn_quantile          7.062       6.259       0.803       1.956       2.509
04_hierarchical_gaussian     7.515       6.627       0.888       2.071       2.774
00_constant                  9.620       8.836       0.784       2.761       2.451
000_do_nothing              27.760       0.018      27.742       0.006      86.695

improvement over today's engine (01_recent_rate):
  02_state_clock               -1.1%
  01_recent_rate               +0.0%
  03_sklearn_quantile          +0.4%
  04_hierarchical_gaussian     +6.9%
  00_constant                 +36.8%
  000_do_nothing             +294.7%


## Is the ranking an artefact of the 10:1 ratio?

Each approach keeps the configuration it chose at 10:1 — re-tuning per ratio would be
a different study — so this shows how robust the *decision* is to the price I picked,
not how well each could do if re-tuned.

In [5]:
sweep = []
for ratio in (1, 3, 10, 30, 100):
    stolen = D.COST_WASTED * ratio
    for name, per_N in predictors.items():
        tot = np.mean([D.cost(test["y%d" % N].values, per_N[N](test),
                              stolen=stolen)["per_anchor"] for N in D.HORIZONS])
        sweep.append({"ratio": "%d:1" % ratio, "model": name, "cost": round(tot, 3)})
S = pd.DataFrame(sweep).pivot(index="model", columns="ratio", values="cost")
S = S[["1:1", "3:1", "10:1", "30:1", "100:1"]]
print(S.to_string())
print("\ncheapest at each ratio:")
for col in S.columns:
    print("  %-6s %s" % (col, S[col].idxmin()))

ratio                        1:1     3:1    10:1    30:1   100:1
model                                                           
000_do_nothing            27.744  27.748  27.760  27.795  27.919
00_constant                1.668   3.435   9.620  27.292  89.143
01_recent_rate             1.860   3.010   7.033  18.529  58.764
02_state_clock             1.595   2.787   6.958  18.875  60.584
03_sklearn_quantile        1.429   2.681   7.062  19.580  63.395
04_hierarchical_gaussian   1.550   2.876   7.515  20.768  67.156

cheapest at each ratio:
  1:1    03_sklearn_quantile
  3:1    03_sklearn_quantile
  10:1   02_state_clock
  30:1   01_recent_rate
  100:1  000_do_nothing


## Where the money is lost, at N = 300

In [6]:
N = 300
y = test["y%d" % N].values
print(D.by_cell(test, y, {n: p[N](test) for n, p in predictors.items()}).to_string(index=False))

 state      band   n  ACTUAL p95  000_do_nothing  00_constant  01_recent_rate  02_state_clock  03_sklearn_quantile  04_hierarchical_gaussian
  idle     night 763        36.0           100.0         22.0             0.0             0.0                  3.3                       9.2
  idle   morning 419        87.0           100.0         22.0             0.0            19.0                 20.5                      20.8
  idle afternoon 379        22.8           100.0         22.0             0.0            25.6                 20.2                      32.3
  idle   evening 444        59.2           100.0         22.0             0.0             2.6                  6.4                      15.1
  warm     night  18         0.0           100.0         22.0             0.0             0.0                  0.0                      19.5
  warm   morning  26        99.5           100.0         22.0             0.0            15.7                 19.4                      27.8
  warm aftern